<h1 style="color:#004080; font-size:2em; margin-bottom:0.3em;"> Accessing NWM Forecasts using Google BigQuery</h1>
<h4 style="color:gray; font-weight:normal;">
  The purpose of this notebook is to access a subset of the data products created by the operational National Water Model.
  A comprehensive description of these data products can be found at
  <a href="https://water.noaa.gov/about/nwm" target="_blank" style="color:#004080; text-decoration:underline;">
    water.noaa.gov/about/nwm
  </a>.
  This notebook leverages a REST API to access National Water Model data stored in Google Cloud Platform's BigQuery service.
  This API was developed by researchers at BYU and is currently hosted by CIROH.
</h4>

<div style="background-color: #f7f7f7; padding: 10px; border-radius: 5px;">
    <p><strong>Authors:</strong> 
    <ul>
        <li>Anthony Castronova, <a href="mailto:acastronova@cuahsi.org">acastronova@cuahsi.org</a></li>
        <li>Irene Garousi-Nejad, <a href="mailto:igarousi@cuahsi.org">igarousi@cuahsi.org</a></li>
    </ul>
    </p>
    <p><strong>Last Modified:</strong> 6/9/25</p>
    <p><strong>Description:</strong> This Jupyter notebook is designed as a tutorial to guide learners through the process of accessing operational National Water Model predictions using the CIROH Big Query API.
    </p>
    <p><strong>Software Requirements:</strong> This notebook has been tested using Python v3.11.8 using the following library versions:
    <blockquote>
        <ul>
            <li>matplotlib==3.8.3</li>
            <li>pandas==2.2.1</li>
            <li>geopandas==1.0.1</li>
            <li>shapely==2.0.3</li>
            <li>ipyleaflet==0.18.2</li>
            <li>sidecar==0.7.0</li>
            <li>ipywidgets==8.1.2</li>
        </ul>
    </blockquote>
    </p>
    <p><strong>Supporting Files and Dependencies:</strong> 
        <ul>
            <li>Credentials to authenticate our queries with the CIROH BigQuery API</li>
            <li>The utils package containing helper functions for retrieving National Water Model time series data using the BigQuery API and for generating interactive maps of model features. </li>
        </ul>
    </p>
    <p><strong>References:</strong></p>
    <ul>
        <li>https://pubs.usgs.gov/publication/sir20255016/full</li>
        <li>https://www.sciencedirect.com/science/article/pii/S1364815224001841</li>
        <li>Banacos, P., 2023, The great Vermont flood of 10–11 July 2023—Preliminary meteorological summary: National Weather Service web page, accessed February 28, 2024, at https://www.weather.gov/btv/The-Great-Vermont-Flood-of-10-11-July-2023-Preliminary-Meteorological-Summary</li>
        <li>https://water.noaa.gov/about/nwm</li>
        <li>https://storymaps.arcgis.com/stories/c4964f08ffcf4d9286bd1fd545ddfbab?play=true&speed=medium</li>
    </ul>
</div>

#### Introduction to CIROH Big Query API and Parameter Setup

We'll be accessing operational National Water Model predictions using the CIROH Big Query API. In order to do so, we'll need to use an access key provided by the Alabama Water Institute (AWI). Instructions for doing so can be found at [DocuHub](https://docs.ciroh.org/docs/products/Data%20Management%20and%20Access%20Tools/bigquery-api/). This capability has been established through CIROH research and a complete description of its design and implementation can be found in the paper entitled: ["Design and implementation of a BigQuery dataset and application programmer interface (API) for the U.S. National Water Model"](https://www.sciencedirect.com/science/article/pii/S1364815224001841).

This project ingests operational NWM predictions, which are stored in NetCDF files, into a cloud optimized tabular structure. This makes a once onerous task of accessing these timeseries data a highly efficient and streamlined process. The general architecture for enabling this capability is outlined in the graphic below:

![BigQuery Architecture](img/bigquery-arch.jpg)

<div style="border-left: 4px solid #007acc; background-color: #eef6fb; padding: 10px; border-radius: 4px;">
    <p> <strong>Summary:</strong>
        This notebook performs the following tasks:
    </p>
    <ol>
        <li>Read Analysis and Assimilation Data</li>
        <li>Read Forecast Data</li>
        <li>Collect Forecast Data for Multiple Reference Times</li>
        <li>Creating and Interactive Map</li>
    </ol>
</div>

<i class="fas fa-code" style="color: black; margin-right: 8px;"></i>
<strong>Import Required Python Libraries and Set Credentials to Authenticate</strong>

In [ ]:
import io
import creds
import pandas
import requests
import numpy as np
import matplotlib.pyplot as plt

We'll use the `creds.py` credentials to authenticate our queries with the CIROH BigQuery API. These need to be passed in the header of each each request we make. Define the header information, which we'll reuse throughout the notebook. 

From the top JupyterHub menu, click on **File → New → Python File**

Once credentials have been obtained by the AWI, they should be saved in a file named `creds.py`. We're saving our credentials in a separate file so that they are not displayed in the notebook and remain a secret. This file should contain two variables:

```
key="<INSERT API KEY HERE>"
url="https://nwm-api.ciroh.org"
```



In [ ]:
API_KEY = creds.key
API_URL = creds.url

header = {
    'x-api-key': API_KEY
}

<i class="fas fa-book-open" style="color: black; margin-right: 8px;"></i>
<strong>Case Study: Flooding in Vermont (July 2023)</strong>

<blockquote style="border-left: 4px solid #007acc; background-color: #f9f9f9; padding: 15px; margin: 20px 0; font-style: italic; color: #333;"> 
<p> <strong>“A major storm caused catastrophic flooding across Vermont between July 9–12, 2023, resulting in millions of dollars in damages.”</strong> The high amount of rainfall caused several rivers to peak at record levels, in some cases surpassing records set during Tropical Storm Irene in 2011. The U.S. Geological Survey, in cooperation with the Federal Emergency Management Agency, collected and analyzed data on peak water-surface elevations, peak streamflow, and annual exceedance probabilities at streamgages, lake gages, and selected ungaged locations. 
</p> 
<p> Of 80 streamgages monitored (with records spanning 12 to 94 years), <strong>11 recorded their highest peak streamflows on record</strong>, and another 10 exceeded the 100-year recurrence interval threshold. Notably, 20 out of 45 monitored sites reported higher peak flows in 2023 than during Irene. 
</p> 
<p> <strong>Seventeen rivers</strong> were surveyed for high-water marks for both events. At 32 of the 103 sites, water levels during the 2023 flood were higher than those in 2011. </p> <p> <strong>Federal assistance exceeded $54.7 million</strong> following a disaster declaration by President Biden. All 14 counties in Vermont received public aid, with 9 also qualifying for individual support. 
</p> 
<p> For more details, see the full report: <a href="https://pubs.usgs.gov/publication/sir20255016/full" target="_blank"> <em>Flood of July 2023 in Vermont (USGS, 2024)</em> </a> 
</p> 


Londonderry, Vermont before and after the storm.

| Before the storm | After the storm |
|---|---|
|![Londonderry Before](img/londonderry-before.png) | ![Londonderry After](img/londonderry-after.png)|


---

![Vermont Storm Rainfall](img/vt-storm.png)

*Figure: Map from the National Oceanic and Atmospheric Administration webpage showing the 48-hour rainfall totals (in inches) from July 10–11, when most of the rainfall occurred, in Vermont and northern New York; from Banacos (2023).*


For our analysis we'll focus on the **West River at South Londonderry**, VT during the period of July 9–12, 2023.

![West River](img/west-river-location.png)

<div style="
    background: linear-gradient(to right, #607d8b, #00bcd4);
    color: white;
    padding: 10px;
    border-radius: 10px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    box-shadow: 0 4px 8px rgba(0,0,0,0.05);
    margin: 0;
">
<i class="fas fa-map-marker-alt" style="color: black; margin-right: 8px;"></i>
<strong>Define the input parameters for our location and period of interest.</p>


In [ ]:
reach_id = 10102374   
start_time = '2023-07-01'
end_time =  '2023-07-20'
output_format = 'csv'

## Read Analysis and Assimilation Data

Build the API request to collect the NWM `analysis and assimilation` predictions. **Analysis and Assimilation** refers to a process that combines observed data with model simulations to produce the most accurate possible estimate of current conditions, known as the model state. These states are saved as restart files for forecast initialization. The model uses a lookback period (the amount of past time that a model considers) to ingest recent observations, improving streamflow estimates. *Note*: Only streamflow is assimilated!

![NWM Anaysis and Assim](img/NWN-StandardAnalysis-CONUS-Configuration.png)

*Figure is developed based on information from https://storymaps.arcgis.com/stories/c4964f08ffcf4d9286bd1fd545ddfbab?play=true&speed=medium*


The CIROH BigQuery API endpoint for the `analysis and assimilation` data accepts the following input parameters:

##### Analysis and Assimilation Input Parameters

<table style="border-collapse: collapse; width: 100%; font-size: 14px;">
  <thead>
    <tr style="background-color: navy; color: white; direction: ltr;">
      <th style="padding: 8px; text-align: left;">Argument</th>
      <th style="padding: 8px; text-align: left;">Type</th>
      <th style="padding: 8px; text-align: left;">Description</th>
    </tr>
  </thead>
  <tbody style="direction: ltr;">
    <tr>
      <td style="padding: 8px; text-align: left;"><code>start_time</code></td>
      <td style="padding: 8px; text-align: left;">str | None</td>
      <td style="padding: 8px; text-align: left;">The start time of the data range. Defaults to <code>None</code>. If None, defaults to <code>"2018-09-17T00:00:00"</code>.</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>end_time</code></td>
      <td style="padding: 8px; text-align: left;">str | None</td>
      <td style="padding: 8px; text-align: left;">The end time of the data range. Defaults to <code>None</code>. If None, defaults to the current time.</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>comids</code></td>
      <td style="padding: 8px; text-align: left;">str | None</td>
      <td style="padding: 8px; text-align: left;">A comma-separated list of reach IDs for the forecast. Defaults to <code>None</code>. Example: <code>"15039097,1239657"</code>.</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>hydroshare_id</code></td>
      <td style="padding: 8px; text-align: left;">str | None</td>
      <td style="padding: 8px; text-align: left;">The HydroShare ID with specified comids to extract the forecast. If <code>comids</code> is not provided, this will be used to extract comids. Defaults to <code>None</code>. Example: <code>"643dc03878704a30849536e302bdb2c0"</code>.</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>output_format</code></td>
      <td style="padding: 8px; text-align: left;">str<</td>
      <td style="padding: 8px; text-align: left;">The format of the analysis-assimilation response data. Defaults to <code>'json'</code>. Supported values: <code>'json'</code>, <code>'csv'</code>.</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>run_offset</code></td>
      <td style="padding: 8px; text-align: left;">int</td>
      <td style="padding: 8px; text-align: left;">The analysis-assim result time offset. Defaults to <code>1</code>. Supported values: <code>1</code>, <code>2</code>, <code>3</code>.</td>
    </tr>
  </tbody>
</table>


In [ ]:
ENDPOINT = f'{API_URL}/analysis-assim'

params = {
    'comids': reach_id,
    'start_time': start_time,
    'end_time': end_time,
    'output_format': output_format,
}

print(ENDPOINT)
print(params)

Call the API using the Python `requests` library to collect data.

In [ ]:
r = requests.get(ENDPOINT,
                 params=params,
                 headers=header)

A Python `requests` object is returned by the previous cell which contains lots of information about the HTTP request and response. One thing that we should always check is the status code of the response, specifically we want to make sure that the request was successful which would result in an `200` code. For more information about HTTP status codes, see this <a href=https://en.wikipedia.org/wiki/List_of_HTTP_status_codes>Wikipedia article</a>.

In [ ]:
if r.status_code == 200:
    print('Request was Sucessful ;)')
else:
    print(f'The request was not successful: {r.status_code}')

Let's preview the response that we recieved from the CIROH BigQuery API. This will be in `csv` format since that's what we specified in the request parameters (i.e. `output_format: csv`). There could be a large amount of data in our response, so let's just view the first 1000 characters so we can see the format of the API response.

In [ ]:
# print the first 1000 characters of the response.
print(r.text[0:1000])

Convert the raw data into a Pandas Dataframe so we can work with it a bit easier.

In [ ]:
df_analysis = pandas.read_csv(io.StringIO(r.text), sep=',')
df_analysis.set_index('time', inplace=True)
df_analysis

We can plot these data very easily using built-in Pandas functions.

In [ ]:
df_analysis.streamflow.plot(grid=True,
                            rot=45,
                            title=f'Streamflow for NWM {reach_id}',
                            ylabel='Streamflow (cms)');

## Read Forecast Data


Forecasts are generated using data from numerical weather prediction models (e.g., HRRR, NBM, CFS) and are initialized using Analysis & Assimilation outputs to simulate future hydrologic conditions such as streamflow and snowmelt.


<blockquote style="border-left: 4px solid #007acc; background-color: #f9f9f9; padding: 15px; margin: 20px 0; font-style: italic; color: #333;"> 
<p> <strong>Short Range</strong> Forced with meteorological data from the HRRR and RAP models, the Short Range Forecast configuration cycles hourly and produces hourly deterministic forecasts of streamflow and hydrologic states out to 18 hours. The model is initialized with a restart file from the Analysis and Assimilation configuration and does not cycle on its own states.
</p> 
<p> <strong>Medium Range</strong> The Medium Range Forecast configuration is executed four times per day, forced with GFS model output. Member 1 extends out to 10 days while members 2-6 extend out to 8.5 days. This configuration produces hourly and 3-hourly deterministic output and is initialized with the restart file from the Analysis and Assimilation configuration.
</p> 
<p> <strong>Long Range</strong> The Long Range Forecast cycles four times per day (i.e. every 6 hours) and produces a daily 16-member 30-day ensemble forecast. There are 4 ensemble members in each cycle of this forecast configuration, each forced with a different CFS forecast member. It produces 6-hourly streamflow and daily land surface output, and, as with the other forecast configurations, is initialized with a common restart file from the Analysis and Assimilation configuration.
</p> 
<p> For more details, see the full report: <a href="https://water.noaa.gov/about/nwm" target="_blank"> <em>Using the National Water Model Within NWPS</em> </a> 
</p> 


![NWM Forecast](img/NWN-ShortMediumLongRanges-CONUS-Configuration.png)

*Figure is developed based on information from https://storymaps.arcgis.com/stories/c4964f08ffcf4d9286bd1fd545ddfbab?play=true&speed=medium*


The CIROH BigQuery API endpoint for `forecast` data takes the following input parameters:

##### Forecast Input Parameters

<table style="border-collapse: collapse; width: 100%; font-size: 14px;">
  <thead>
    <tr style="background-color: navy; color: white; direction: ltr;">
      <th style="padding: 8px; text-align: left;">Argument</th>
      <th style="padding: 8px; text-align: left;">Type</th>
      <th style="padding: 8px; text-align: left;">Description</th>
    </tr>
  </thead>
  <tbody style="direction: ltr;">
    <tr>
      <td style="padding: 8px; text-align: left;"><code>forecast_type</code></td>
      <td style="padding: 8px; text-align: left;">str</td>
      <td style="padding: 8px; text-align: left;">The forecast run to extract data from. Supported values are <code>'long_range'</code>, <code>'medium_range'</code>, and <code>'short_range'</code>.</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>reference_time</code></td>
      <td style="padding: 8px; text-align: left;">str | None</td>
      <td style="padding: 8px; text-align: left;">The reference time for the forecast. If <code>None</code>, defaults to the latest available forecast reference time in the specified table. Example: <code>"2023-11-25 06:00:00 UTC"</code>.</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>comids</code></td>
      <td style="padding: 8px; text-align: left;">str | None</td>
      <td style="padding: 8px; text-align: left;">A comma-separated list of reach IDs for the forecast. Defaults to <code>None</code>. Example: <code>"15039097,1239657"</code>.</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>hydroshare_id</code></td>
      <td style="padding: 8px; text-align: left;">str | None</td>
      <td style="padding: 8px; text-align: left;">The HydroShare ID with specified comids to extract the forecast. If <code>comids</code> is not provided, this will be used to extract comids. Defaults to <code>None</code>. Example: <code>"643dc03878704a30849536e302bdb2c0"</code>.</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>ensemble</code></td>
      <td style="padding: 8px; text-align: left;">str | None</td>
      <td style="padding: 8px; text-align: left;">A comma-separated list of ensembles for the forecast. If <code>None</code>, the average of all available ensembles is used. Supported values: <code>0</code> for <code>short_range</code>, <code>0–5</code> for <code>medium_range</code>, and <code>0–3</code> for <code>long_range</code>.</td>
    </tr>
    <tr>
      <td style="padding: 8px; text-align: left;"><code>output_format</code></td>
      <td style="padding: 8px; text-align: left;">str</td>
      <td style="padding: 8px; text-align: left;">The output format for the forecast dataset. Defaults to <code>'json'</code>. Supported values are <code>'json'</code> and <code>'csv'</code>.</td>
    </tr>
  </tbody>
</table>

Define input parameters for our request.

In [ ]:
forecast_type = 'medium_range'
ensembles = '0,1,2,3,4,5,6'
reference_time = '2023-07-05'

Build the API query object.

In [ ]:
ENDPOINT = f'{API_URL}/forecast'

params = {
    'comids': reach_id,
    'forecast_type':  forecast_type,
    'reference_time': reference_time,
    'ensemble': ensembles,
    'output_format': output_format,
}

Call the API using the Python `requests` library to collect data.

In [ ]:
r = requests.get(ENDPOINT,
                 params=params,
                 headers=header)

if r.status_code == 200:
    print('Request was Sucessful ;)')
else:
    print(f'The request was not successful: {r.status_code}')

Convert the raw data into a Pandas Dataframe.

In [ ]:
df = pandas.read_csv(io.StringIO(r.text), sep=',')
df.set_index('time', inplace=True)
df

We forecasts generated from multiple ensemble members. Let's plot these on the same axis so we can see how they differ.

In [ ]:
df.groupby('ensemble')['streamflow'].plot(grid=True,
                                          rot=45,
                                          legend=True,
                                          title=f'Forecasted Streamflow for NWM {reach_id} Grouped by Ensemble');

Another way to visualize these data is by plotting their interquartile range. This will help us visualize theses collection of forecasts behave over time, as a shaded area with an upper bound of the 75th percentile and lower bound of the 25th percentile. This is effective in reducing visual clutter while also showing the agreement/disagreement between each forecast. A wide band of shaded area represents disagreement (or higher uncertainty) between series, where as a narrow shaded area represents agreement (or low uncertainty).

In [ ]:

iqr = df.groupby(df.index)['streamflow'].quantile([0.25, 0.75])
iqr = iqr.reset_index()
iqr = iqr.rename(columns={'level_1': 'quantile'})

df_pivot = iqr.pivot(index='time', columns='quantile', values='streamflow')
df_pivot.index = pandas.to_datetime(df_pivot.index)

# Plot the time series for each group
plt.figure(figsize=(10, 6))
plt.fill_between(df_pivot.index, df_pivot[0.25], df_pivot[0.75], color='gray', alpha=0.3);
plt.xticks(rotation=25)

plt.grid(True)
plt.title(f'IQR of Forecasted Streamflow for NWM {reach_id}')
plt.ylabel('Streamflow (cms)')
plt.xlabel('Date')
plt.tight_layout()

## Collect Forecast Data for Multiple Reference Times

We can also collect data for multiple forecasts through time. To assist with this, we'll use a utility library called `utils.forecast`. This library wraps the forecast request above in the `concurrent.futures` class which enables us to make parallel requests. This will speed up the collection of data.

In [ ]:
from utils import forecast
from datetime import datetime, timedelta

Instantiate the utilities library by passing it our BigQuery key.

In [ ]:
forecasts = forecast.Forecasts(creds.key)

Define several reach ids to collect data for. In this scenario, we're only interested in the first ensemble member.

In [ ]:
comids = [10102374, 10102316, 10102406]
forecast_type = 'medium_range'
ensembles = [0,1,2,3,4,5,6]

Create a list of times for which data will be collected. In the cell below, we define a start time and dynamically generate a list of dates that will be sent to the CIROH BigQuery API.

In [ ]:
# start time 
start_date = datetime(2023, 6, 28)

# Number of days to increment (for example, 10 days)
num_days = 20

# Create a list of dates incrementing by day
reference_times = [(start_date + timedelta(days=i)).strftime('%Y-%m-%d') for i in range(num_days)]

print("Generated Reference Times:")
for reference_time in reference_times:
    print(reference_time)

Collect medium_range forecasts for each of the `comids` specied above, for each of the times generated in the previous cell.

In [ ]:
forecasts.collect_forecasts(comids,
                            forecast_type,
                            ensembles,
                            reference_times)

Let's double check that we received forecast data for multiple reaches and multiple days.

In [ ]:
for fid in forecasts.df.feature_id.unique():
    num_timesteps = len(forecasts.df.loc[forecasts.df.feature_id==fid].time.unique())
    num_ref_times = len(forecasts.df.loc[forecasts.df.feature_id==fid].reference_time.unique())
    print(f'Reach ID: {fid}')
    print(f'\tNumber of Ref Times: {num_ref_times}')
    print(f'\tNumber of Timesteps: {num_timesteps}\n')

Preview the data for one of these reaches.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for reference_time, group in forecasts.df.groupby('ensemble'):
    group.plot(x='time', y='streamflow', ax=ax, label=str(reference_time), legend=False, color='k', linestyle="", marker='.')

ax.grid(True)
ax.set_title(f'Forecasted Streamflow for NWM {reach_id} from 6/29/23 - 7/17/23')
ax.set_ylabel('Streamflow (cms)')
ax.set_xlabel('Date')
plt.show()

When we're looking at many timeseries, it can sometimes be helpful to visualize the interquartile range. This descibes the middle 50% of the values at each timestep and shows were the forecasts align. 

In [ ]:
# compute IQR for the ensembles of each forecast.
dats = []
for ref_time, forecast_grp in forecasts.df.groupby('reference_time'):
    iqr = forecast_grp.groupby(forecast_grp.time)['streamflow'].quantile([0.25, 0.75])
    iqr = iqr.reset_index()
    iqr = iqr.rename(columns={'level_1': 'quantile'})
    df_pivot = iqr.pivot(index='time', columns='quantile', values='streamflow')
    df_pivot.index = pandas.to_datetime(df_pivot.index)
    dats.append(df_pivot)

For a single forecast, this would look something like this:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(pandas.to_datetime(df_analysis.index),
        df_analysis['streamflow'],
        label='Analysis Streamflow',
        color='black')


ax.fill_between(dats[0].index,
                    dats[0][0.25],
                    dats[0][0.75],
                    color='blue',
                    alpha=0.1,
                    zorder=2);

ax.collections[-1].set_label("Medium Range Forecast IQR")
ax.grid(True, color='lightgray', alpha=0.6, zorder=1)
ax.set_ylabel('Streamflow (cms)')
ax.set_xlabel('Date')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()

If we plot the IQR for all the forecast series that we collected in a semi-transparent manner, the areas appearing darker show us where the forecasts are in agreement.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(pandas.to_datetime(df_analysis.index),
        df_analysis['streamflow'],
        label='Analysis Streamflow',
        color='black')
for i in range(0, len(dats)):
    ax.fill_between(dats[i].index,
                    dats[i][0.25],
                    dats[i][0.75],
                    color='blue',
                    alpha=0.1,
                    zorder=2);

ax.collections[-1].set_label("Medium Range Forecast IQR")
ax.grid(True, color='lightgray', alpha=0.6, zorder=1)
ax.set_ylabel('Streamflow (cms)')
ax.set_xlabel('Date')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()

## Creating and Interactive Map

We've included a utility library to demonstrate how these CIROH BigQuery API capabilities can be used in an interactive mapping interface. Reading through the code for producing this map interface is left as an exercise for the reader to complete. The relevant code can be found in the `ForecastMap` class within `./utilities/forecast.py`

In [ ]:
f = forecast.ForecastMap(creds.key)
f.asSideCarMap()